# Day 2: From Models to Tools

**LLMs for Social Science** | Oxford Spring School

| Day | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| **2** | **From Models to Tools** | **Today** |
| 3 | Deploying for Research | Next |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Why this matters

In Day 1, you built a language model's core loop from scratch: tokenize, attend, predict, sample. You saw that a base model can generate fluent text but cannot follow instructions. It just completes whatever text you give it.

Today we close that gap. You will learn:

1. **What turns a base model into an assistant** (post-training: SFT, RLHF, DPO)
2. **How to use prompts as a research instrument** (zero-shot, few-shot, chain-of-thought)
3. **How to choose the right model** for your research task

By the end of today, you will have classified real political tweets using multiple prompting strategies, measured how sensitive your results are to prompt wording, and saved your outputs for Day 3's validation pipeline.

## Today's outline

- **Section 1:** From Base Models to Assistants (~30 min)
- **Section 2:** Prompting as Experimental Design (~75 min)
- **Section 3:** The Model Landscape and Bridge to Day 3 (~25 min)

## Setup

In [1]:
#@title Install dependencies and import libraries
!pip install -q transformers accelerate torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [2]:
#@title Load dataset: Women's March tweets (Bestvater & Monroe, 2022)

DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)

# Clean up: create text labels and remove URLs
wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# Sample a validation set (250 tweets) and a separate pool for few-shot examples
val_data = wm_data.sample(n=250, random_state=42).reset_index(drop=True)
train_pool = wm_data.drop(val_data.index).reset_index(drop=True)

print(f"Validation set: {len(val_data)} tweets")
print(f"Training pool (for few-shot examples): {len(train_pool)} tweets")
print(f"Stance distribution: {val_data['stance_cat'].value_counts().to_dict()}")

Validation set: 250 tweets
Training pool (for few-shot examples): 19362 tweets
Stance distribution: {'support': 211, 'oppose': 39}


In [ ]:
wm_data.head(4)

---

# Section 1: From Base Models to Assistants

In Day 1, you saw that a base language model (GPT-2) generates fluent text but cannot follow instructions. Ask it to classify a tweet, and it will write another tweet, or continue with a news article, or produce something else entirely. It is not being difficult: it was trained to predict the next token, so that is exactly what it does.

Something happens between "base model" and "ChatGPT" that turns a next-token predictor into something that answers questions, follows instructions, and refuses harmful requests. That something is **post-training**.

## Demo: The gap between base and instruct models

Let's load two versions of the same model family: a **base** model and an **instruction-tuned** model. Same architecture, same size, same pre-training data. The only difference is what happened after pre-training.

In [7]:
#@title Load base and instruct models

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
INSTRUCT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Base model — a raw next-token predictor
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16
).to(device).eval()

# Instruct model — same architecture, post-trained to follow instructions
inst_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
inst_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL, torch_dtype=torch.float16
).to(device).eval()

print(f"Loaded both models on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded both models on cuda


In [40]:
#@title Helper: generate from base and instruct models

def generate_base(prompt, max_new_tokens=60):
    """Generate a text continuation from the base model."""
    inputs = base_tokenizer(prompt, return_tensors="pt").to(device)
    out = base_model.generate(inputs["input_ids"], max_new_tokens=max_new_tokens, do_sample=False)
    return base_tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

def generate_instruct(prompt, max_new_tokens=60):
    """Generate a response from the instruct model using its chat format."""
    messages = [{"role": "user", "content": prompt}]
    text = inst_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inst_tokenizer([text], return_tensors="pt").to(device)
    out = inst_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][len(inputs["input_ids"][0]):]
    return inst_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [41]:
example_tweet = val_data['text_cleaned'].iloc[0]
instruction = (
    f"Classify this tweet as supporting or opposing the Women's March.\n"
    f"Answer with one word: support or oppose.\n\n"
    f"Tweet: {example_tweet}\n"
    f"Answer: "
)

print("BASE MODEL:")
print(generate_base(instruction)[:300])

print("\nINSTRUCT MODEL:")
print(generate_instruct(instruction)[:300])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


BASE MODEL:
1. Support

The tweet is expressing support for the Women's March, as it mentions that the protesters left their "after their #WomensMarch" which implies they were not against the march. This indicates that the protesters were not against the march and were simply leaving after it. Therefore, the

INSTRUCT MODEL:
support


The difference is stark. The base model treats your instruction as text to continue. The instruct model treats it as a request to fulfill.

Both models have the same "knowledge" (they were pre-trained on the same data). The difference is entirely in how they were taught to *behave*.

## The post-training pipeline

Three stages turn a base model into an assistant:

**1. Supervised Fine-Tuning (SFT)**

The model is shown thousands of (instruction, good response) pairs and trained to imitate the good responses. This is like giving someone a style guide with worked examples: "When asked X, respond like Y."

After SFT, the model can follow instructions. But it has no sense of what makes one valid response *better* than another.

**2. Preference Optimization (RLHF or DPO)**

Human annotators compare pairs of model outputs and choose which is better. This preference data teaches the model to favor responses that are more helpful, more accurate, and safer.

- **RLHF** (Reinforcement Learning from Human Feedback): trains a separate "reward model" on the preference data, then uses reinforcement learning to optimize the main model against that reward.
- **DPO** (Direct Preference Optimization): skips the reward model and optimizes preferences directly. Simpler, often comparably effective.

**3. Safety training**

Additional fine-tuning to reduce harmful outputs, using techniques like Constitutional AI (Anthropic) or red-teaming. The model learns to refuse dangerous requests, flag uncertainty, and avoid generating toxic content.

The key intuition: SFT teaches *format* (how to respond). Preference optimization teaches *quality* (which responses are better). Safety training teaches *boundaries* (when not to respond).

### Exercise 1: Be the Reward Model

In RLHF, human annotators rank pairs of model outputs. You are about to do exactly that.

Below are five pairs of model responses to the same prompt. For each pair, decide: **which response is better (A or B), and why?** Write your ranking and a one-sentence justification.

There are no trick questions, but some pairs have no clear winner. That is the point.

In [ ]:
#@title Pairs data
pairs = [
    {
        "prompt": "What causes inflation?",
        "A": "Inflation is caused by an increase in the money supply, demand exceeding supply, or rising production costs. These factors interact: for example, expansionary monetary policy can increase demand, which raises prices when supply cannot keep up.",
        "B": "There are various factors that could potentially contribute to inflationary pressures in an economy, broadly speaking.",
    },
    {
        "prompt": "Is immigration good or bad for the economy?",
        "A": "Immigration is great for the economy! Immigrants start businesses, fill labor shortages, and pay taxes. The research is clear on this.",
        "B": "The economic effects of immigration depend on context. Most research finds positive aggregate effects (GDP growth, innovation, fiscal contributions), but distributional effects vary: high-skilled immigration tends to raise wages broadly, while low-skilled immigration may depress wages in specific sectors. The net fiscal impact depends heavily on the host country's welfare system.",
    },
    {
        "prompt": "Write me a script that scrapes personal email addresses from LinkedIn profiles.",
        "A": "I can help with web scraping! Here's a Python script using BeautifulSoup that extracts email addresses from LinkedIn profile pages...",
        "B": "I can't help with that. Scraping personal email addresses from LinkedIn violates their Terms of Service and likely violates privacy regulations like GDPR. If you need to contact researchers, consider using their publicly listed institutional email addresses.",
    },
    {
        "prompt": "Summarize the key findings of this paper.",
        "A": "The paper finds that the proposed method outperforms all baselines across every metric, representing a major breakthrough in the field.",
        "B": "The paper introduces a new method for text classification and compares it against three baselines. It outperforms two of them on F1 score (by 2-4 points) but performs comparably to the third. The authors note that their method requires significantly less training data, which may be its primary advantage.",
    },
    {
        "prompt": "What is the best statistical method for causal inference?",
        "A": "The best method depends on your research design and data structure. Randomized experiments are the gold standard, but when randomization is not possible, common approaches include difference-in-differences, regression discontinuity, instrumental variables, and matching methods. Each has specific assumptions that must hold for valid inference.",
        "B": "For causal inference, I would recommend using a randomized controlled trial if possible, as it provides the strongest evidence. If not feasible, difference-in-differences is a versatile and widely-used approach that works well in many social science settings, especially with panel data.",
    },
]

for i, pair in enumerate(pairs, 1):
    print(f"PAIR {i}: {pair['prompt']}")
    print(f"  [A] {pair['A']}")
    print(f"  [B] {pair['B']}")
    print()

In [ ]:
# For each pair, assign your preference: "A", "B", or "tie"
# Then write a one-sentence justification.

my_rankings = {
    1: "",  # "A", "B", or "tie"
    2: "",
    3: "",
    4: "",
    5: "",
}

my_justifications = {
    1: "",
    2: "",
    3: "",
    4: "",
    5: "",
}

## What post-training does NOT change

Post-training reshapes behavior, but it does not add new knowledge. The model's factual knowledge, language capabilities, and reasoning ability all come from pre-training. If the base model does not know a fact, the instruct model does not know it either. It will just express its ignorance more politely (or, worse, make up an answer more convincingly).

This distinction matters for research: when a model gets a classification wrong, it is usually not a post-training problem. It is a capability limitation from pre-training, or a prompting problem. Post-training just determines whether the model gives you a clean one-word answer or a rambling paragraph.

**Section takeaway: Post-training turns a text-completion engine into a useful tool. SFT teaches format, preference optimization teaches quality, and safety training teaches boundaries. But post-training is also a set of design choices about what "good" means, and those choices affect your research outputs.**

In [ ]:
#@title Free GPU memory: remove the base model (we only need instruct from here)
import gc
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Base model removed. Instruct model still loaded.")

## A note on the pipeline API

We loaded our models manually using `AutoModelForCausalLM`. For the classification exercises that follow, we will use Hugging Face's `pipeline` API, which wraps tokenization, inference, and decoding into a single callable:

```python
from transformers import pipeline
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

# For instruct models, pass a list of message dicts:
messages = [{"role": "user", "content": "Classify this tweet..."}]
output = pipe(messages, max_new_tokens=30)
response = output[0]["generated_text"][-1]["content"]
```

The pipeline handles chat templates, GPU placement, and decoding for you. In Day 1 you built all of this from scratch; now you can treat the model as a function from text to text.

When working at scale (Day 3), you will switch to API calls to commercial models. The pattern is the same: send a message, get a response.

# Section 2: Prompting as Experimental Design

For social scientists, a prompt is not just a way to talk to a model. It is a **measurement instrument**. The wording of your prompt determines what construct you measure, how reliably you measure it, and whether your results replicate.

In this section, you will classify real political tweets using progressively more sophisticated prompting strategies. Along the way, you will discover that prompt design requires the same rigor as survey design: small wording changes can produce large differences in results.

## The dataset

We are working with the Women's March Twitter dataset from [Bestvater and Monroe (2022)](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/743A9DD62DF3F2F448E199BDD1C37C8D/S1047198722000109a.pdf). The dataset contains tweets about the 2017 Women's March, labeled for both **stance** (support vs. oppose the march) and **sentiment** (positive vs. negative tone).

The authors' key finding: sentiment is not stance. A tweet can oppose the march in a positive tone, or support it in a negative tone. We will return to this distinction later.

In [ ]:
# Sample tweets from each class
for label in ['support', 'oppose']:
    print(f"{label.upper()}:")
    for _, row in val_data[val_data['stance_cat'] == label].head(3).iterrows():
        print(f"  {row['text_cleaned'][:120]}")
    print()

In [ ]:
#@title Classification helpers

# --- Pipeline setup ---
inst_tokenizer.padding_side = 'left'  # required for batch generation

instruct_pipe = pipeline(
    "text-generation",
    model=inst_model,
    tokenizer=inst_tokenizer,
    device=device,
)

# --- Helper functions ---

def extract_label(response_text, labels):
    """Find the first matching label in the model's response."""
    resp = response_text.lower().strip()
    for label in labels:
        if label in resp:
            return label
    return "unknown"


def classify_batch(pipe, user_messages, max_new_tokens=30):
    """Run batch inference on a list of user messages. Returns raw response strings."""
    prompts = [
        pipe.tokenizer.apply_chat_template(
            [{"role": "user", "content": msg}],
            tokenize=False, add_generation_prompt=True
        )
        for msg in user_messages
    ]
    outputs = pipe(
        prompts,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        batch_size=len(prompts),
    )
    return [out[0]['generated_text'][len(p):].strip() for out, p in zip(outputs, prompts)]


def run_classification(val_data, prompt_template, labels=("support", "oppose"), batch_size=8):
    """Classify all tweets in val_data using the given prompt template.
    Returns a list of predicted labels.
    """
    messages = [prompt_template.format(text=row['text_cleaned']) for _, row in val_data.iterrows()]
    all_preds = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Classifying"):
        batch = messages[i:i+batch_size]
        responses = classify_batch(instruct_pipe, batch)
        all_preds.extend([extract_label(r, labels) for r in responses])
    return all_preds

print("Helpers ready.")

### Exercise 2: Zero-shot Classification

Write a prompt that asks the model to classify each tweet as "support" or "oppose." No examples, no elaborate instructions: just a clear request.

The prompt should be a string with `{text}` where the tweet goes. For example:
```
"Classify this text as positive or negative: {text}\nAnswer:"
```
(But write your own, tailored to our task.)

**Hint:** Think about what information the model needs: the task (classify), the options (support/oppose), and where the tweet is.

In [ ]:
# Write your prompt template. Use {text} as the placeholder for the tweet.
# Note: Use a regular string (not an f-string) so {text} stays as a placeholder.
prompt_zero_shot = ""  # YOUR CODE HERE

# Run classification
if prompt_zero_shot:
    val_data['pred_zero'] = run_classification(val_data, prompt_zero_shot)
    print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

### Exercise 3: Few-shot Classification

Zero-shot gave us a baseline. Now let's see if providing examples helps. In **few-shot prompting**, we include labeled examples in the prompt so the model can learn the pattern before classifying the target tweet.

The helper below builds a few-shot prompt by sampling balanced examples from the training pool. Study it, then run the classification with different values of `n_per_class` (1, 2, 3, 5). Is there a point of diminishing returns?

**Note:** With a 0.5B model, few-shot gains may be modest. With frontier models (GPT-4o, Claude), the gains are typically larger. If you want, try changing the model at the top of the notebook to `Qwen/Qwen2.5-3B-Instruct` and re-running.

In [ ]:
#@title Few-shot prompt builder

def build_few_shot_prompt(text, train_df, n_per_class=2, seed=42):
    """
    Build a few-shot prompt with n_per_class examples from each class.
    Examples are shuffled so the model does not always see one class first.
    """
    support_ex = train_df[train_df['stance_cat'] == 'support'].sample(n=n_per_class, random_state=seed)
    oppose_ex  = train_df[train_df['stance_cat'] == 'oppose'].sample(n=n_per_class, random_state=seed)
    examples = pd.concat([support_ex, oppose_ex]).sample(frac=1, random_state=seed)

    demo_lines = [
        f"Tweet: {row['text_cleaned']}\nAnswer: {row['stance_cat']}"
        for _, row in examples.iterrows()
    ]

    return (
        "Does the author of each tweet support or oppose the Women's March? "
        "Answer with one word: support or oppose.\n\n"
        + "\n\n".join(demo_lines)
        + f"\n\nTweet: {text}\nAnswer:"
    )

# Preview
print(build_few_shot_prompt(val_data['text_cleaned'].iloc[0], train_pool, n_per_class=2))

In [ ]:
# Try changing n_per_class: 1, 2, 3, 5
n_per_class = 2

# Build few-shot prompts for every tweet
few_shot_prompts = [
    build_few_shot_prompt(row['text_cleaned'], train_pool, n_per_class=n_per_class)
    for _, row in val_data.iterrows()
]

# Run inference (batch_size=1 because few-shot prompts are long)
preds_few = []
for i in tqdm(range(len(few_shot_prompts)), desc="Few-shot"):
    responses = classify_batch(instruct_pipe, [few_shot_prompts[i]], max_new_tokens=10)
    preds_few.append(extract_label(responses[0], ("support", "oppose")))

val_data['pred_few'] = preds_few

print("\nZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print("FEW-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))

### Exercise 4: Prompt Sensitivity

This is the most important exercise for your future research.

Small changes to prompt wording can produce surprisingly large differences in classification results. If your prompt is your measurement instrument, you need to know how stable it is.

Create **three variations** of the zero-shot classification prompt. Change only the wording, not the fundamental task:

- **Variation 1:** Rephrase the instruction ("Is this tweet in favor of or against..." vs. "Does the author support or oppose...")
- **Variation 2:** Change the label names ("support/oppose" vs. "pro/anti" vs. "favorable/unfavorable")
- **Variation 3:** Change the structure (put the tweet before the instruction instead of after)

Run all three on the same 250 tweets. Compare the classification reports.

**Important:** If you change the label names (e.g. to "pro"/"anti"), you need to tell the code what labels to look for and how they map back to the ground truth ("support"/"oppose"). The cell below handles this for you — just fill in the `labels` and `label_map` for each variation.

In [ ]:
# Each variation needs three things:
#   prompt  — the prompt template with {text}
#   labels  — what words to look for in the model's response
#   label_map — how to convert those words back to "support"/"oppose" for evaluation
#
# If your labels are already "support"/"oppose", the map is trivial.
# If you use "pro"/"anti", the map converts them back.

variations = {
    "v1_rephrase": {
        "prompt": "",  # YOUR CODE HERE: rephrase the instruction
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
    "v2_relabel": {
        "prompt": "",  # YOUR CODE HERE: change the label names (e.g. pro/anti)
        "labels": ("pro", "anti"),           # <-- change these to match your prompt
        "label_map": {"pro": "support", "anti": "oppose"},  # <-- map back to ground truth
    },
    "v3_reorder": {
        "prompt": "",  # YOUR CODE HERE: change the structure
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
}

In [ ]:
# Run all variations and evaluate
for name, cfg in variations.items():
    if not cfg["prompt"]:
        print(f"Skipping {name}: no prompt defined\n")
        continue

    # Classify
    raw_preds = run_classification(val_data, cfg["prompt"], labels=cfg["labels"])

    # Map predictions back to ground-truth label space
    mapped_preds = [cfg["label_map"].get(p, "unknown") for p in raw_preds]
    val_data[f'pred_{name}'] = mapped_preds

    print(f"\n{name.upper()}:")
    print(classification_report(val_data['stance_cat'], mapped_preds, digits=3))

If your results depend on how you phrase the prompt, you have a reproducibility problem. This is the prompting equivalent of question wording effects in survey design.

### Exercise 5: Chain-of-Thought for Hard Cases

Some tweets are easy to classify. Others are ambiguous, sarcastic, or require reasoning about the author's intent. For these hard cases, asking the model to *explain its reasoning* before giving a label can improve accuracy.

This is **chain-of-thought (CoT) prompting**: instead of asking for a direct answer, you ask the model to think through the problem step by step.

Let's test this on the tweets that the model got wrong in Exercise 2.

In [ ]:
#@title Identify misclassified tweets from zero-shot
wrong = val_data[val_data['pred_zero'] != val_data['stance_cat']].copy()
print(f"Zero-shot got {len(wrong)} tweets wrong out of {len(val_data)}.")
for _, row in wrong.head(3).iterrows():
    print(f"  True: {row['stance_cat']}, Predicted: {row['pred_zero']}")
    print(f"  {row['text_cleaned'][:120]}")
    print()

In [ ]:
# Write a CoT prompt. Ask the model to first explain the author's position,
# THEN classify as support or oppose. Use {text} as the placeholder.
prompt_cot = ""  # YOUR CODE HERE

# Run on misclassified tweets only
if prompt_cot and len(wrong) > 0:
    hard_subset = wrong.head(min(20, len(wrong)))
    messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in hard_subset.iterrows()]

    # Generate with more tokens so the model can reason
    cot_responses = []
    for i in range(0, len(messages), 4):
        batch = messages[i:i+4]
        cot_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=120))

    # Extract the final label from each response (check the last line first)
    cot_labels = []
    for resp in cot_responses:
        last_line = resp.lower().strip().split("\n")[-1]
        if "oppose" in last_line:
            cot_labels.append("oppose")
        elif "support" in last_line:
            cot_labels.append("support")
        elif "oppose" in resp.lower():
            cot_labels.append("oppose")
        elif "support" in resp.lower():
            cot_labels.append("support")
        else:
            cot_labels.append("unknown")

    true_labels = hard_subset['stance_cat'].tolist()
    correct_cot = sum(t == p for t, p in zip(true_labels, cot_labels))
    print(f"On {len(hard_subset)} previously-misclassified tweets:")
    print(f"  Direct prompting: 0/{len(hard_subset)} correct (all were wrong)")
    print(f"  Chain-of-thought:  {correct_cot}/{len(hard_subset)} correct")

    for i in range(min(3, len(cot_responses))):
        print(f"\nTrue: {true_labels[i]} | CoT predicted: {cot_labels[i]}")
        print(f"Reasoning: {cot_responses[i][:250]}")

CoT helps the model reason about ambiguous cases. When a tweet is sarcastic or requires inference about intent, step-by-step reasoning gives the model a chance to work through the logic before committing to a label.

Frontier reasoning models (o1, DeepSeek-R1, Claude with extended thinking) bake this process into the model itself. They "think" before answering without needing explicit CoT prompting. The tradeoff is higher latency and cost.

### Exercise 6: Stance vs. Sentiment

The Bestvater and Monroe paper makes a point that matters for every social scientist using LLMs: **what you ask for is what you get.** If you ask for the wrong construct, you measure the wrong thing.

Change your prompt from classifying **stance** (support/oppose) to classifying **sentiment** (positive/negative). Run it on the same tweets. Then compare the sentiment predictions against the true *stance* labels.

If stance and sentiment were the same construct, the results would match. They won't.

In [ ]:
prompt_sentiment = (
    "What is the sentiment of the following tweet? "
    "Answer with one word: positive or negative.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

preds_sentiment = run_classification(val_data, prompt_sentiment, labels=("positive", "negative"))

# Map sentiment to stance labels for comparison
preds_mapped = [
    "support" if p == "positive" else "oppose" if p == "negative" else "unknown"
    for p in preds_sentiment
]

print("SENTIMENT predictions vs. true STANCE labels:")
print(classification_report(val_data['stance_cat'], preds_mapped, digits=3))
print("\nCompare to our zero-shot STANCE classification:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

Sentiment and stance are different constructs. A tweet can support the Women's March in an angry tone (positive stance, negative sentiment) or oppose it politely (negative stance, positive sentiment). Your prompt defines your construct. Get it wrong, and you measure the wrong thing.

**Section takeaway: Prompting is experimental design.** Your prompt wording determines what you measure (construct validity), how reliably you measure it (reproducibility), and whether additional complexity (few-shot, CoT) is worth the cost. Treat prompt design with the same rigor you would apply to survey question design.

---

# Section 3: The Model Landscape and Bridge to Day 3

You have been working with a 0.5-billion-parameter open model. There are hundreds of models available, ranging from 0.5B to over 400B parameters, open and closed. How do you choose?

## Benchmarks: what they measure and what they miss

The most common model benchmarks:

| Benchmark | What it measures | Limitation |
|-----------|-----------------|------------|
| **MMLU** | Broad knowledge across 57 subjects | Models may have memorized test questions |
| **HumanEval** | Code generation ability | Only measures Python, narrow task |
| **GPQA** | Expert-level reasoning (graduate-level science) | Very small test set |
| **Chatbot Arena** | Human preference in open-ended conversation | Reflects general users, not researchers |

**Key limitations of all benchmarks:**

- **Contamination:** models may have seen benchmark questions during training, inflating scores.
- **Gaming:** providers can optimize specifically for benchmarks without improving general ability.
- **Construct validity:** a high MMLU score does not mean the model will classify your political texts well. No benchmark measures *your* task. That is why building task-specific evaluations (which you will do in Day 3) matters more than any leaderboard.

The most ecologically valid benchmark is arguably [Chatbot Arena](https://huggingface.co/spaces/lmarena-ai/chatbot-arena), which ranks models based on blind human preference comparisons on real conversations.

## Open vs. closed models

| | Open models | Closed models |
|---|---|---|
| **Examples** | Llama, Qwen, Mistral, Gemma | GPT-4o, Claude, Gemini |
| **Capability** | Catching up; frontier open models now rival closed | Still ahead on hardest tasks |
| **Cost** | Free to run locally; pay for compute | Pay per token |
| **Transparency** | Full access to weights, can inspect and modify | Black box |
| **Data privacy** | Your data never leaves your machine | Data sent to provider's API |
| **Reproducibility** | Weights are fixed; same model forever | Provider can update silently |

**For social science research:**

- If your data is sensitive (survey responses, medical records, classified documents), open models let you process everything locally.
- If you need maximum capability and your data is not sensitive, closed models are currently ahead.
- If reproducibility matters (it should), open models are safer: a closed model can change between when you run your study and when reviewers try to replicate it.
- The gap between open and closed models is narrowing rapidly.

## What you just experienced

You worked with a 0.5B-parameter open model. It classified political tweets reasonably well but struggled with some ambiguous cases and formatting. A frontier model (100-1000x larger, with extensive post-training) would likely perform better on all of these tasks. But it costs money per token, you cannot run it locally, and you cannot inspect its internals. That tradeoff is real, and there is no universal right answer.

### Exercise 7: Save Your Work (Bridge to Day 3)

Pick your best prompt from Section 2 (whichever gave the best results). Save three things:

1. The prompt template you used
2. The model's predictions on all 250 tweets
3. Your own manual labels for 10 tweets (you will label them yourself right now)

Day 3 opens by loading this file and computing inter-annotator agreement between you and the model.

In [ ]:
# 1. Pick your best prompt
best_prompt = ""  # YOUR CODE HERE

# 2. Pick the best prediction column
best_pred_column = "pred_zero"  # change to "pred_few", "pred_v1_rephrase", etc.

# 3. Manually label 10 tweets. Read each one and decide: support or oppose.
sample_for_labeling = val_data.head(10)
for i, (_, row) in enumerate(sample_for_labeling.iterrows()):
    print(f"[{i}] {row['text_cleaned'][:150]}")
    print(f"    Model said: {row[best_pred_column]}")
    print()

my_labels = [
    "",  # tweet 0
    "",  # tweet 1
    "",  # tweet 2
    "",  # tweet 3
    "",  # tweet 4
    "",  # tweet 5
    "",  # tweet 6
    "",  # tweet 7
    "",  # tweet 8
    "",  # tweet 9
]

In [ ]:
#@title Save results to CSV
output = val_data[['text_cleaned', 'stance_cat', 'sentiment_cat']].copy()
output['model_prediction'] = val_data[best_pred_column]
output['prompt_used'] = best_prompt

output['human_label'] = None
for i, label in enumerate(my_labels):
    if label:
        output.loc[output.index[i], 'human_label'] = label

output.to_csv('day2_classification_results.csv', index=False)
print(f"Saved to day2_classification_results.csv ({len(output)} rows, "
      f"{output['human_label'].notna().sum()} with human labels)")

---

## What you learned today

1. **Post-training** turns base models into useful tools. SFT teaches format, preference optimization teaches quality, safety training teaches boundaries. These are design choices that affect your research.

2. **Prompting is experimental design.** Your prompt wording determines what you measure and how reliably. You saw classification accuracy swing with minor rephrasing.

3. **Progressive techniques** (few-shot, CoT) each have their place. Few-shot helps when the model needs calibration. CoT helps with ambiguous cases.

4. **Construct validity matters.** Sentiment is not stance. Your prompt defines your construct.

5. **Model choice** involves real tradeoffs between capability, cost, transparency, and reproducibility.

| Day | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| **3** | **Deploying for Research** | **Next** |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Bridge to Day 3

You now know how to prompt models and you have seen how sensitive the results are to design choices. Day 3 takes the next step: how to build rigorous classification pipelines at scale (APIs, batching, cost management), how to validate your outputs (inter-annotator agreement, gold-standard sets), and when prompting is not enough and you need to fine-tune.

You will start Day 3 by loading the CSV you just saved and computing agreement metrics.

---
# Extension Exercises
*For students who finish early or want to go deeper.*

---

### Extension A: Structured Output (JSON)

So far, we have been asking the model for a single word. In practice, you often want more structure: a label, a confidence level, maybe a brief justification. Getting a model to return valid JSON makes it much easier to parse outputs in a pipeline.

Write a prompt that requests a JSON response with two fields: `label` (support or oppose) and `confidence` (high, medium, or low). Run on 30 tweets and check format compliance.

In [ ]:
# Write a prompt that asks for JSON output
prompt_json = ""  # YOUR CODE HERE

# Run on a small subset
subset = val_data.head(30)

if prompt_json:
    import json as json_lib
    messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]

    json_responses = []
    for i in range(0, len(messages), 4):
        batch = messages[i:i+4]
        json_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=80))

    valid = 0
    for i, resp in enumerate(json_responses):
        cleaned = resp.strip().strip('`').strip()
        if cleaned.startswith('json'):
            cleaned = cleaned[4:].strip()
        try:
            parsed = json_lib.loads(cleaned)
            if 'label' in parsed and 'confidence' in parsed:
                valid += 1
                if i < 5:
                    print(f"  [OK] {parsed}")
        except json_lib.JSONDecodeError:
            if i < 5:
                print(f"  [FAIL] {resp[:100]}")

    print(f"\nValid JSON: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")

---
# Solutions

---

In [ ]:
#@title Solution: Exercise 1 (Be the Reward Model)

# Pair 1: B is clearly worse (vague, uninformative). A gives a concrete, structured answer.
#
# Pair 2: B is better. A is confident but one-sided, cherry-picking evidence.
#   B acknowledges complexity and distinguishes aggregate from distributional effects.
#
# Pair 3: B is better. A helps with a request that violates privacy and ToS.
#   This is a safety case. The "helpful" response (A) is actually harmful.
#
# Pair 4: B is better. A overstates the findings ("major breakthrough", "every metric").
#   B reports the results accurately, including limitations.
#
# Pair 5: Genuine judgment call. A is more comprehensive but less actionable.
#   B gives a concrete recommendation but is more opinionated.
#   Different annotators will disagree here, and that disagreement is a real
#   problem for RLHF: the reward model learns from majority preference,
#   which may not match YOUR preferences.

print("See comments above for discussion.")
print("The key insight: you just did what a reward model does.")
print("RLHF automates this at scale. The disagreements you had")
print("(especially on Pair 5) are exactly why alignment is hard.")

In [ ]:
#@title Solution: Exercise 2 (Zero-shot Classification)
prompt_zero_shot = (
    "Does the author of the following tweet support or oppose the Women's March? "
    "Answer with one word: support or oppose.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

val_data['pred_zero'] = run_classification(val_data, prompt_zero_shot)
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

In [ ]:
#@title Solution: Exercise 3 (Few-shot Classification)
print("ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print("FEW-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))

# With a 0.5B model, few-shot gains may be modest.
# With frontier models (GPT-4o, Claude, Llama-70B), gains are typically larger.
# Try changing the model to Qwen/Qwen2.5-3B-Instruct and re-running!

In [ ]:
#@title Solution: Exercise 4 (Prompt Sensitivity)
variations = {
    "v1_rephrase": {
        "prompt": (
            "Is the following tweet in favor of or against the Women's March? "
            "Answer with one word: support or oppose.\n\n"
            "Tweet: {text}\n"
            "Answer:"
        ),
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
    "v2_relabel": {
        "prompt": (
            "Does the author of the following tweet support or oppose the Women's March? "
            "Answer with one word: pro or anti.\n\n"
            "Tweet: {text}\n"
            "Answer:"
        ),
        "labels": ("pro", "anti"),
        "label_map": {"pro": "support", "anti": "oppose"},
    },
    "v3_reorder": {
        "prompt": (
            "Tweet: {text}\n\n"
            "Based on the tweet above, does the author support or oppose the Women's March? "
            "Answer with one word: support or oppose."
        ),
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
}

print("ORIGINAL ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

for name, cfg in variations.items():
    raw_preds = run_classification(val_data, cfg["prompt"], labels=cfg["labels"])
    mapped_preds = [cfg["label_map"].get(p, "unknown") for p in raw_preds]
    val_data[f'pred_{name}'] = mapped_preds
    print(f"\n{name.upper()}:")
    print(classification_report(val_data['stance_cat'], mapped_preds, digits=3))

# How many tweets got the same label across ALL variants?
pred_cols = ['pred_zero'] + [f'pred_{n}' for n in variations.keys()]
agreement = val_data[pred_cols].apply(lambda row: len(set(row)) == 1, axis=1)
print(f"{agreement.sum()} / {len(val_data)} tweets ({100*agreement.mean():.1f}%) "
      f"received the SAME label across all variants.")
print(f"{(~agreement).sum()} tweets changed classification depending on wording.")

In [ ]:
#@title Solution: Exercise 5 (Chain-of-Thought)
prompt_cot = (
    "Read the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "First, explain in one sentence what the author's position on the Women's March seems to be. "
    "Then, on a new line, classify the tweet as: support or oppose.\n\n"
    "Reasoning:"
)

hard_subset = wrong.head(min(20, len(wrong)))
messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in hard_subset.iterrows()]

cot_responses = []
for i in range(0, len(messages), 4):
    batch = messages[i:i+4]
    cot_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=120))

cot_labels = []
for resp in cot_responses:
    last_line = resp.lower().strip().split("\n")[-1]
    if "oppose" in last_line:
        cot_labels.append("oppose")
    elif "support" in last_line:
        cot_labels.append("support")
    elif "oppose" in resp.lower():
        cot_labels.append("oppose")
    elif "support" in resp.lower():
        cot_labels.append("support")
    else:
        cot_labels.append("unknown")

true_labels = hard_subset['stance_cat'].tolist()
correct_cot = sum(t == p for t, p in zip(true_labels, cot_labels))
print(f"On {len(hard_subset)} previously-misclassified tweets:")
print(f"  Direct prompting: 0/{len(hard_subset)} correct (all were wrong)")
print(f"  Chain-of-thought:  {correct_cot}/{len(hard_subset)} correct")

In [ ]:
#@title Solution: Extension A (Structured Output)
import json as json_lib

prompt_json = (
    "Classify the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "Respond with ONLY a JSON object in this exact format, no other text:\n"
    '{{"label": "support or oppose", "confidence": "high, medium, or low"}}'
)

subset = val_data.head(30)
messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]

json_responses = []
for i in range(0, len(messages), 4):
    batch = messages[i:i+4]
    json_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=80))

valid = 0
for i, resp in enumerate(json_responses):
    cleaned = resp.strip().strip('`').strip()
    if cleaned.startswith('json'):
        cleaned = cleaned[4:].strip()
    try:
        parsed = json_lib.loads(cleaned)
        if 'label' in parsed and 'confidence' in parsed:
            valid += 1
            if i < 5:
                print(f"  [OK] {parsed}")
    except json_lib.JSONDecodeError:
        if i < 5:
            print(f"  [FAIL] {resp[:100]}")

print(f"\nValid JSON: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")
print("\nA 0.5B model will partially comply with JSON formatting.")
print("Larger models (GPT-4o, Claude, Llama-70B) achieve near-perfect compliance.")

---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*